# Thermal-Contrast — what the U-Net actually sees

For elements of `TermoDataset`, show the five channels produced by
`extract_channels(video) -> (5, H, W)` and the frames that were rejected on the way.

| channel | formula | meaning |
|---|---|---|
| `maxmin` | max(t) − min(t) | thermal contrast, classic NDT defect map |
| `std` | σ over t | how strongly a pixel moves during the run |
| `pca1` | 1st temporal EOF over **every kept frame** | dominant heating mode (Chulkov Seq 7) |
| `tsr_d1` | max \|d/d ln(t)\| of 5th-order log-log poly fit | 1st TSR derivative (Chulkov Seq 5) |
| `tsr_d2` | max \|d²/d ln(t)²\| of 5th-order log-log poly fit | 2nd TSR derivative (Chulkov Seq 6) |

In [ ]:
# === configuration ===
INCLUDE = None          # None -> every sub-dataset; or e.g. ["dataset_tpu"]
NUM_FRAMES = 64         # frames sampled for maxmin / std
TSR_POLY_DEGREE = 5     # 5th-order log-log polynomial (Chulkov et al., 2019)
PREVIEW = ["R_002", "R_009", "Z_007", "sample3", "sample6", "sample12"]

In [ ]:
import importlib
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models" / "Thermal-Contrast").is_dir())
TC_DIR = str(ROOT / "models" / "Thermal-Contrast")
if TC_DIR not in sys.path:
    sys.path.insert(0, TC_DIR)

import paths  # noqa: F401  puts repo root and models/ on sys.path

import matplotlib.pyplot as plt
import numpy as np
import torch

import channels as channels_mod
importlib.reload(channels_mod)
from channels import (
    CHANNEL_NAMES,
    CHANNEL_TITLES,
    NUM_CHANNELS,
    ChannelParams,
    build_channels,
    compute_tsr_derivatives,
    select_frames,
)
from data import source_name, video_id
from datasets import TermoDataset

PARAMS = ChannelParams(num_frames=NUM_FRAMES, tsr_poly_degree=TSR_POLY_DEGREE)
dataset = TermoDataset(root_dir=str(paths.DATASETS_ROOT), include=INCLUDE)
index_of = {video_id(path): i for i, (path, _, _) in enumerate(dataset.items)}

print(f"{len(dataset)} videos, {NUM_CHANNELS} channels, params key {PARAMS.key}")
print("channels:", ", ".join(CHANNEL_NAMES))
for name in sorted({source_name(path) for path, _, _ in dataset.items}):
    ids = [video_id(p) for p, _, _ in dataset.items if source_name(p) == name]
    print(f"  {name}: {len(ids)} videos, {ids[0]} .. {ids[-1]}")

## 1. Frame rejection

Per-frame brightness and spatial spread decide which frames are used. A **flash**
frame is bright everywhere and spatially flat; **leading** frames are a run at
`t = 0` whose spread is far above the pre-heating baseline (camera calibration
frames), which would otherwise make `t₀` meaningless.

In [ ]:
def show_frame_selection(name: str) -> None:
    video, _ = dataset[index_of[name]]
    sel = select_frames(video, PARAMS)

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
    steps = np.arange(video.shape[0])
    for axis, values, label in zip(axes, [sel.mean, sel.std], ["frame mean", "frame spatial \u03c3"]):
        axis.plot(steps, values.numpy(), lw=0.8)
        axis.axvline(sel.onset, color="tab:green", ls="--", lw=1, label=f"heating onset t={sel.onset}")
        for flag, colour, tag in [(sel.flash, "tab:red", "flash"), (sel.leading, "tab:orange", "leading")]:
            hit = flag.nonzero().flatten().numpy()
            if hit.size:
                axis.scatter(hit, values.numpy()[hit], s=26, color=colour, zorder=3, label=f"{tag} ({hit.size})")
        axis.scatter(sel.sampled.numpy(), values.numpy()[sel.sampled.numpy()], s=6, color="k", zorder=2, label="sampled")
        axis.set_title(f"{name} \u2014 {label}")
        axis.set_xlabel("frame")
        axis.legend(fontsize=7)
    fig.tight_layout()
    plt.show()
    print(sel.summary())


for name in ["R_002", "sample3"]:
    show_frame_selection(name)

## 2. Channels on the U-Net input

Top row: a few raw frames spanning the run. Bottom row: the five channels plus the
ground-truth mask. Bright regions in the channels should line up with the white
boxes in `GT`.

TSR channels use a 5th-order log-log polynomial fit on the cooling phase (same
idea as `thermo.features.tsr` / `irt_data.features.TSRCoeffsFeatureExtractor`).

## 2a. TSR derivatives (before normalization)

Cooling-phase excess temperature → 5th-order log-log polynomial → max |d/d ln(t)| and
max |d²/d ln(t)²| per pixel. Same pipeline as `thermo.features.tsr` /
`irt_data.features.TSRCoeffsFeatureExtractor`, but collapsed to two scalar maps.

In [ ]:
def show_tsr(name: str) -> None:
    video, _ = dataset[index_of[name]]
    sel = select_frames(video, PARAMS)
    kept = video[sel.keep.nonzero().flatten()]
    d1, d2 = compute_tsr_derivatives(kept, PARAMS)

    fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.6))
    for axis, arr, title in zip(
        axes,
        [d1, d2],
        [CHANNEL_TITLES["tsr_d1"] + " (raw)", CHANNEL_TITLES["tsr_d2"] + " (raw)"],
    ):
        image = axis.imshow(arr.numpy(), cmap="inferno")
        axis.set_title(title, fontsize=10)
        axis.axis("off")
        fig.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    fig.suptitle(
        f"{name} — TSR on {sel.num_kept} kept frames, poly deg={PARAMS.tsr_poly_degree}",
        fontsize=10,
    )
    fig.tight_layout()
    plt.show()


for name in ["R_002", "sample3"]:
    show_tsr(name)

In [ ]:
def show_channels(name: str, *, raw_frames: int = 5) -> torch.Tensor:
    video, mask = dataset[index_of[name]]
    sel = select_frames(video, PARAMS)
    channels = build_channels(video, sel, PARAMS)
    truth = (mask > 0).numpy()

    picks = sel.sampled[torch.linspace(0, sel.sampled.numel() - 1, raw_frames).long()]
    columns = max(raw_frames, len(CHANNEL_NAMES) + 1)
    fig, axes = plt.subplots(2, columns, figsize=(2.6 * columns, 5.6))
    for axis in axes.ravel():
        axis.axis("off")

    for column, frame in enumerate(picks.tolist()):
        axes[0, column].imshow(video[frame].numpy(), cmap="inferno")
        axes[0, column].set_title(f"raw t={frame}", fontsize=9)

    for column, channel_name in enumerate(CHANNEL_NAMES):
        axes[1, column].imshow(channels[column].numpy(), cmap="inferno", vmin=0, vmax=1)
        axes[1, column].set_title(CHANNEL_TITLES[channel_name], fontsize=10)
    axes[1, len(CHANNEL_NAMES)].imshow(truth, cmap="gray", vmin=0, vmax=1)
    axes[1, len(CHANNEL_NAMES)].set_title("GT", fontsize=10)

    fig.suptitle(f"{name}  \u2014  {sel.summary()}", fontsize=10)
    fig.tight_layout()
    plt.show()
    return channels


extracted = {name: show_channels(name) for name in PREVIEW if name in index_of}

## 3. Are the channels informative?

Contrast-to-noise is `|mean(inside GT) − mean(outside GT)| / σ(outside GT)`.

Read it as a rough ranking between channels of the *same* video, not as an absolute
detectability score. The denominator includes the dark bands at the frame edges and
the hot rig hardware, whose spread is an order of magnitude larger than the
pixel-to-pixel variation on the specimen itself, so the ratio lands near 1.0 even for
videos whose defect grid is obvious in the images above.

In [ ]:
def contrast_to_noise(channel: torch.Tensor, gt: torch.Tensor) -> float:
    inside, outside = channel[gt], channel[~gt]
    return float((inside.mean() - outside.mean()).abs() / outside.std().clamp_min(1e-9))


header = f"{'video':10s} " + " ".join(f"{n:>10s}" for n in CHANNEL_NAMES)
print(header)
print("-" * len(header))
for name, channels in extracted.items():
    _, mask = dataset[index_of[name]]
    gt = mask > 0
    scores = [contrast_to_noise(channels[c], gt) for c in range(len(CHANNEL_NAMES))]
    print(f"{name:10s} " + " ".join(f"{s:10.2f}" for s in scores))